# PIPE 2 — Retriever Analytics (Evaluation & Diagnostics)

Run this after PIPE 1. It loads the FAISS index + docs mapping, evaluates retrieval-space metrics, and exports diagnostic artifacts.

## Run order
- 0) Setup & configuration
- 1) Load required artifacts from previous pipe(s)
- 2) Run computations
- 3) Export tables/figures/logs to runs/ and artifacts/

## Notes
- This release contains **no embedded secrets** and **no stored outputs**.
- Configure paths and keys via environment variables or `.env`.


# 🔭 SciBERT-SolarPhysics-Search — Colab Pipeline

Pipeline reproducível (célula a célula) para treinar e publicar um **sentence encoder**
especializado em *solar physics / MHD / PIML / (multi)modal transformers* a partir
das bases deduplicadas **Núcleo**, **PIML** e **CombFinal**, e projetar no corpus **ML-108k**.

**Etapas:**
1) Setup e logging · 2) Leitura/concatenação · 3) DAPT (MLM) no SciBERT
4) Fine-tuning contrastivo (Sentence-Transformers) · 5) Clusters + rótulos fracos
6) Projeção ML-108k + FAISS · 7) Métricas e relatórios · 8) (Opcional) Publicação HF

> **Observação:** este notebook **não** refaz deduplicação. As bases fornecidas já estão
tratadas. Ajuste `DATA_DIR`/nomes de arquivos conforme seus caminhos no Drive/Colab.

In [ ]:
#@title Instalação de pacotes (execute uma vez)
!pip -q install --upgrade pip
!pip -q install pandas numpy scikit-learn matplotlib tqdm ipywidgets
!pip -q install transformers datasets accelerate sentence-transformers
!pip -q install umap-learn hdbscan faiss-cpu
!pip -q install yake keybert
!pip -q install pyreadr
import os, sys, json, math, re, time, random, pathlib
print('✅ Pacotes instalados')

In [ ]:
#@title Configurações gerais e helper de logging
from pathlib import Path
from datetime import datetime
RUN_NAME = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
BASE_DIR = Path('/content')
RUN_DIR = BASE_DIR / f'runs/{RUN_NAME}_SciBERT_SolarPhysics'
LOG_DIR = RUN_DIR / 'logs'
ART_DIR = RUN_DIR / 'artifacts'
CKPT_DIR = RUN_DIR / 'ckpts'
REP_DIR = RUN_DIR / 'reports'
for d in [LOG_DIR, ART_DIR, CKPT_DIR, REP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def log_write(filename: str, line: str):
    f = LOG_DIR / filename
    ts = datetime.utcnow().isoformat()+'Z'
    msg = f'[{filename.replace(".log","")}] {ts} | ' + line
    print(msg)
    with open(f, 'a', encoding='utf-8') as fp:
        fp.write(msg + '\n')

print(f'📁 RUN_DIR = {RUN_DIR}')

In [ ]:
#@title Caminhos de dados (ajuste conforme seus arquivos no Colab)
from pathlib import Path

# Eles estão na raiz do ambiente Colab
DATA_DIR = Path('/content')

# Use os nomes exatos dos arquivos que aparecem na barra lateral
NUCLEO = DATA_DIR / 'Nucleo_bibliometrix.RData'
PIML   = DATA_DIR / 'PIML_bibliometrix.RData'
COMBF  = DATA_DIR / 'CombFinal_bibliometrix.RData'
ML108K = DATA_DIR / 'ML_Scopus_bibliometrix.RData'

print('Arquivos esperados:')
print(NUCLEO)
print(PIML)
print(COMBF)
print(ML108K)


In [ ]:
#@title Leitura robusta (CSV ou RData) + concatenação `text_full`
import pandas as pd, numpy as np
import unicodedata, re
import pyreadr

def read_bibliometrix(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == '.csv':
        df = pd.read_csv(path, encoding='utf-8', low_memory=False)
    elif path.suffix.lower() == '.rdata' or path.suffix.lower() == '.rda':
        r = pyreadr.read_r(str(path))
        df = list(r.values())[0]
    else:
        raise ValueError(f'Formato não suportado: {path}')
    # Normaliza colunas típicas
    df.columns = [c.strip().upper() for c in df.columns]
    # Campos ausentes que usaremos
    for c in ['TI','AB','DE','ID','PY','DI','SO','AU','TC']:
        if c not in df.columns:
            df[c] = ''
    return df

def clean_text(s: str) -> str:
    if not isinstance(s, str): s = str(s)
    s = s.replace('\n',' ').replace('\r',' ')
    s = re.sub(r'https?://\S+|doi:\S+',' ', s, flags=re.I)
    s = s.lower().strip()
    s = ''.join(ch for ch in unicodedata.normalize('NFKD', s)
                if not unicodedata.combining(ch))
    return re.sub(r'\s+',' ', s)

def concat_text(df: pd.DataFrame) -> pd.DataFrame:
    for col in ['TI','AB','DE','ID']:
        df[col] = df[col].fillna('').astype(str).apply(clean_text)
    df['text_full'] = df['TI'].str.strip() + '. ' + df['AB'].str.strip()
    df['text_full'] = df['text_full'] + ' keywords: ' + df['DE'] + '; ' + df['ID']
    return df

dfs = {}
for name, path in [('nucleo',NUCLEO),('piml',PIML),('combf',COMBF),('ml',ML108K)]:
    df = read_bibliometrix(path)
    df = concat_text(df)
    out = ART_DIR / f'concat_{name}.csv'
    df.to_csv(out, index=False)
    dfs[name] = df
    log_write('01_prep.log', f'base={name} | in={len(df)} | dedup=skipped | out={len(df)}')

print('✅ Concatenação concluída. Arquivos em:', ART_DIR)

In [ ]:
#@title DAPT (MLM) — SciBERT em Núcleo + PIML + CombFinal (sem ML-108k)
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)
import torch, math, json

model_name = "allenai/scibert_scivocab_uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# --- Criação do dataset de treino ---
mlm_texts = pd.concat(
    [dfs["nucleo"]["text_full"], dfs["piml"]["text_full"], dfs["combf"]["text_full"]],
    ignore_index=True,
)
mlm_ds = Dataset.from_dict({"text": mlm_texts.tolist()})

def tok(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=384)

tok_ds = mlm_ds.map(tok, batched=True, remove_columns=["text"])
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=True, mlm_probability=0.15
)

model_mlm = AutoModelForMaskedLM.from_pretrained(model_name)

# --- TrainingArguments (corrigido p/ Transformers ≥5.x) ---
args = TrainingArguments(
    output_dir=str(CKPT_DIR / "scibert_dapt"),
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    num_train_epochs=2,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=100,
    save_strategy="epoch",
    eval_strategy="epoch",       # <- novo nome
    fp16=True,                   # mixed precision
    report_to="none"             # evita warning do wandb
)

trainer = Trainer(
    model=model_mlm,
    args=args,
    data_collator=data_collator,
    train_dataset=tok_ds,
    eval_dataset=tok_ds.select(range(min(4000, len(tok_ds))))
)

# --- Treinamento + logs ---
log_write("02_mlm_dapt.log", f"start | samples={len(tok_ds)} | model={model_name}")
trainer.train()
metrics = trainer.evaluate()

ppl = math.exp(metrics["eval_loss"]) if metrics.get("eval_loss") else None
with open(REP_DIR / "mlm_eval.txt", "w") as fp:
    fp.write(json.dumps(metrics, indent=2))
    fp.write(f"\nperplexity≈{ppl}\n")

log_write("02_mlm_dapt.log", f"end | eval_loss={metrics.get('eval_loss'):.4f} | ppl≈{ppl:.2f}")

best_dir = CKPT_DIR / "scibert_dapt_best"
trainer.save_model(best_dir)
tokenizer.save_pretrained(best_dir)

print("✅ DAPT finalizado e salvo em:", best_dir)


In [ ]:
#@title Fine-tuning Contrastivo (Sentence-Transformers)
# Desativar COMPLETAMENTE o Weights & Biases no Colab
import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "dryrun"
os.environ["DISABLE_MLFLOW_INTEGRATION"] = "true"
print("✅ wandb completamente desativado")

from sentence_transformers import SentenceTransformer, models, losses, InputExample
from torch.utils.data import DataLoader
import random

# carrega backbone do DAPT
word = models.Transformer(str(CKPT_DIR / 'scibert_dapt_best'))
pool = models.Pooling(word.get_word_embedding_dimension(), pooling_mode_mean_tokens=True, pooling_mode_cls_token=False, pooling_mode_max_tokens=False)
norm = models.Normalize()
st_model = SentenceTransformer(modules=[word, pool, norm])

def build_pairs(df: pd.DataFrame, max_n=20000):
    pairs = []
    cols = ['TI','AB','DE','ID']
    df = df.dropna(subset=['AB','TI']).copy()
    sample = df.sample(n=min(max_n, len(df)), random_state=42)
    for _, r in sample.iterrows():
        a = str(r['TI'])
        b = str(r['AB'])
        if len(a)>10 and len(b)>10:
            pairs.append(InputExample(texts=[a,b]))
        if isinstance(r['DE'], str) and len(r['DE'])>5:
            pairs.append(InputExample(texts=[str(r['DE']), b]))
        if isinstance(r['ID'], str) and len(r['ID'])>5:
            pairs.append(InputExample(texts=[str(r['ID']), b]))
    random.shuffle(pairs)
    return pairs

pairs = []
pairs += build_pairs(dfs['nucleo'], 10000)
pairs += build_pairs(dfs['piml'],   12000)
pairs += build_pairs(dfs['combf'],   4000)
log_write('03_contrastive_ft.log', f'pairs_built={len(pairs)}')

loader = DataLoader(pairs, batch_size=64, shuffle=True, drop_last=True)
loss = losses.MultipleNegativesRankingLoss(st_model)
st_model.fit(
    train_objectives=[(loader, loss)],
    epochs=2,
    warmup_steps=int(0.1*len(loader)),
    show_progress_bar=True,
    output_path=str(CKPT_DIR / 'SciBERT-SolarPhysics-Search')
)
log_write('03_contrastive_ft.log', 'training_done')
print('✅ Modelo salvo em', CKPT_DIR / 'SciBERT-SolarPhysics-Search')

In [ ]:
#@title Clustering (HDBSCAN) + rótulos fracos + resumo de clusters
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
import umap
import hdbscan
import numpy as np

st = SentenceTransformer(str(CKPT_DIR / 'SciBERT-SolarPhysics-Search'))
dom_texts = pd.concat([dfs['nucleo']['text_full'], dfs['piml']['text_full'], dfs['combf']['text_full']], ignore_index=True).tolist()
batch = 256
embs = []
for i in range(0, len(dom_texts), batch):
    embs.append(st.encode(dom_texts[i:i+batch], normalize_embeddings=True))
embs = np.vstack(embs)

um = umap.UMAP(n_neighbors=50, min_dist=0.0, metric='cosine', random_state=42).fit_transform(embs)
clusterer = hdbscan.HDBSCAN(min_cluster_size=50, metric='euclidean')
labels = clusterer.fit_predict(um)
pd.DataFrame({'cluster':labels}).to_csv(ART_DIR/'clusters_domain.csv', index=False)

# Rotulagem simples por TF-IDF dos textos por cluster
df_dom = pd.DataFrame({'text':dom_texts, 'cluster':labels})
top_terms = []
for c in sorted(set(labels)):
    if c == -1: continue
    sub = df_dom[df_dom['cluster']==c]['text']
    vec = TfidfVectorizer(max_features=5000, ngram_range=(1,2), stop_words='english')
    X = vec.fit_transform(sub)
    idx = np.argsort(X.sum(axis=0).A.ravel())[::-1][:10]
    terms = [list(vec.vocabulary_.keys())[list(vec.vocabulary_.values()).index(i)] for i in idx]
    top_terms.append({'cluster':int(c),'keyphrases':', '.join(terms)})
pd.DataFrame(top_terms).to_csv(REP_DIR/'clusters_summary.csv', index=False)
log_write('04_clusters_labeling.log', f'clusters={len(set(labels))- (1 if -1 in set(labels) else 0)}')
print('✅ Clusters e rótulos (simples) gerados')

In [ ]:
#@title Normalização extra (remove boilerplate/editorial e marcas textuais)
import re

BOILERPLATE = [
    r'\belsevier\b.*?\bright(s)?\s+reserved\b',   # elsevier all rights reserved
    r'\ball\s+rights\s+reserved\b',
    r'\b©\s*\d{4,}\b', r'\b(copyright|©)\b',
]

INLINE_TAGS = [
    r'\binf\s*/\s*inf\b', r'\bsup\s*/\s*sup\b', r'\bsub\s*/\s*sub\b',
    r'\b(et|al)\.\b',
]

def strong_clean(s: str) -> str:
    s = str(s or '')
    s = re.sub(r'https?://\S+|doi:\S+', ' ', s, flags=re.I)
    for pat in BOILERPLATE + INLINE_TAGS:
        s = re.sub(pat, ' ', s, flags=re.I)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

# Aplique na base 'base' usada na geração de rótulos:
for c in ['TI','AB','DE','ID']:
    base[c] = base[c].map(strong_clean)
base['text_full'] = base['TI'] + '. ' + base['AB'] + ' keywords: ' + base['DE'] + '; ' + base['ID']
print('✅ Limpeza forte aplicada.')


In [ ]:
#@title Léxicos EN-only ampliados (Fenômeno/Técnica/…)
import re, pandas as pd
from pathlib import Path

# --------- PHENOMENA (solar/plasma/MHD) ---------
fenomeno_patterns = {
    'solar_flare': r'\bsolar\s+flare(s)?\b|\b(goes[-\s]?(x|m|c)\s*class)\b|\bflare\s+forecast(ing)?\b',
    'solar_wind': r'\bsolar\s+wind\b|\binterplanetary\s+magnetic\s+field\b|\bimf\b',
    'magnetic_field_flux': r'\bmagnetic\s+field(s)?\b|\bmagnetic\s+flux\b|\bmagnetogram(s)?\b',
    'magnetic_reconnection': r'\bmagnetic\s+reconnection\b',
    'sunspot_active_region': r'\bsunspot(s)?\b|\bactive\s+region(s)?\b|\bar[\s\-]?\d{3,5}\b',
    'coronal_hole': r'\bcoronal\s+hole(s)?\b',
    'cme': r'\bcoronal\s+mass\s+ejection(s)?\b|\bcme(s)?\b|\bshock\s+(front|arrival)\b',
    'corona_heliosphere': r'\bcorona(l)?\b|\bheliosphere\b|\bmagnetosphere\b|\bheliospheric\s+current\s+sheet\b',
    'mhd_plasma_turbulence': r'\bmhd\b|\bmagnetohydrodynamic(s)?\b|\bplasma\s+turbulence\b|\balfv[eé]n(\s+wave(s)?)?\b',
    'plasma_diffusion_transport': r'\b(plasma\s+)?diffusion\b|\btransport\s+process(es)?\b|\badvection[-\s]?diffusion\b',
    'coronal_loop': r'\bcoronal\s+loop(s)?\b|\bloops?\s+oscillation(s)?\b',
    'chromosphere_photosphere': r'\bchromosphere\b|\bphotosphere\b|\bquiet\s+sun\b',
}

# --------- TASKS ---------
tarefa_patterns = {
    'forecasting': r'\bforecast(ing)?\b|\bnowcast(ing)?\b|\bprediction\b',
    'classification': r'\bclassification\b|\bclassifier(s)?\b',
    'detection': r'\bdetection\b|\bdetect(ion|or|ing)?\b',
    'segmentation': r'\bsegmentation\b',
    'regression': r'\bregression\b|\binverse\s+problem(s)?\b',
}

# --------- METHODS (PIML / physics-based) ---------
metodo_patterns = {
    'piml_pinn': r'physics[-\s]?informed|physics[-\s]?guided|theory[-\s]?guided|\bpinn(s)?\b|pde[-\s]?constrained|physics[-\s]?constrained',
    'governing_equations': r'\b(maxwell|navier[-\s]?stokes|poisson|helmholtz|mhd|advection[-\s]?diffusion)\b\s*(equation|eqs|equations)?',
    'hybrid_physics_ml': r'\bhybrid\s+(physics|model|approach)\b|\bdata[-\s]?driven\s+and\s+physics[-\s]?based\b',
    'statistical_bayesian': r'\bbayesian\b|\bstatistical\b',
}

# --------- TECHNIQUES (Transformers + viz/seq/graph baselines) ---------
tecnica_patterns = {
    # Transformers & attention
    'transformer': r'\btransformer(s)?\b|transformer[-\s]*(encoder|decoder)\b|\btransformer[-\s]?based\b',
    'attention_variants': r'\b(self|cross|multi[-\s]?head)\s*attention\b|\battention\s+mechanism\b|\btemporal\s+fusion\s+transformer\b|\btft\b',
    'vit_vision_transformer': r'\bvision\s+transformer\b|\bvit\b|\bswin[-\s]?transformer\b',
    'multimodal_fusion': r'\bmultimodal\b|\bmulti[-\s]?modal\b|\b(data|feature)\s+fusion\b|\bearly\s+fusion\b|\blate\s+fusion\b|\bcross[-\s]?modal\b',
    # Baselines úteis para contraste
    'cnn': r'\bcnn(s)?\b|\bconvolutional\s+neural\s+network(s)?\b|\bresnet\b|\befficientnet\b',
    'lstm_gru_rnn': r'\blstm\b|\bgru\b|\brnn(s)?\b|\brecurrent\s+neural\s+network\b',
    'gnn_graph': r'\bgnn(s)?\b|\bgraph\s+neural\s+network(s)?\b|\bgraph\s+convolution\b',
    'svm_rf_xgb': r'\bsvm\b|\bsupport\s+vector\s+machine\b|\brandom\s+forest\b|\bxgboost\b|\bxgb\b',
    'autoencoder_vae': r'\bautoencoder(s)?\b|\bvariational\s+autoencoder\b|\bvae\b',
}

# --------- DATA / INSTRUMENTS ---------
dados_patterns = {
    'magnetogram_hmi': r'\bmagnetogram(s)?\b|\bhmi\b|\bsdo/hmi\b',
    'aia_imagery': r'\baia\b|\bsdo/aia\b',
    'goes': r'\bgoes(-\w+)?\b',
    'rhessi': r'\brhessi\b',
    'soho_lasco': r'\bsoho\b|\blasco\b',
    'stereo': r'\bstereo\b',
    'hinode': r'\bhinode\b',
    'parker_solar_probe': r'\bparker\s+solar\s+probe\b|\bpsp\b',
}

# Extensão Fenômeno
fenomeno_patterns.update({
    'solar_energetic_particles': r'\bsolar\s+energetic\s+particle(s)?\b|\bsep(s)?\b',
    'magnetic_helicity_instability': r'\bmagnetic\s+helicity\b|\b(plasma|magnetic)\s+instabilit(y|ies)\b',
    'shock_acceleration': r'\bshock\s+(wave|acceleration|front)\b',
    'coronal_heating': r'\bcoronal\s+heating\b',
    'alfven_wave': r'\balfv[eé]n(\s+wave(s)?)?\b',
    'plasma_wave_dynamics': r'\bplasma\s+wave(s)?\b|\bwave\s+dissipation\b',
    'parker_spiral_probe': r'\bparker\s+(spiral|solar\s+probe)\b|\bpsp\b',
})

# Extensão Técnica
tecnica_patterns.update({
    'ann_dnn_ffn': r'\bartificial\s+neural\s+network(s)?\b|\bdeep\s+neural\s+network(s)?\b|\bfeed[-\s]?forward\s+network(s)?\b',
    'ensemble_learning': r'\bensemble\s+learning\b|\bmodel\s+ensemble(s)?\b',
    'multi_task_learning': r'\bmulti[-\s]?task\s+learning\b|\bmulti[-\s]?objective\s+learning\b',
    'spatio_temporal_transformer': r'\bspatio[-\s]?temporal\s+(transformer|network)\b|\btemporal\s+fusion\s+transformer\b',
    'dual_encoder': r'\bdual[-\s]?encoder\b|\bbiencoder\b',
    'gat_graph_attention': r'\bgraph\s+attention\s+network(s)?\b|\bganet\b|\bgat\b',
    'physics_constrained_loss': r'\bphysics[-\s]?constrained\s+loss\b|\bphysical\s+constraint(s)?\b',
})


def tag_with_patterns(text, patterns_dict):
    hits = []
    for label, pat in patterns_dict.items():
        if re.search(pat, text, flags=re.I):
            hits.append(label)
    return ';'.join(sorted(set(hits)))

out = pd.DataFrame({'DOI': base.get('DI',''), 'PY': base.get('PY','')})
txt = base['text_full']

out['Fenomeno'] = txt.map(lambda s: tag_with_patterns(s, fenomeno_patterns))
out['Tarefa']   = txt.map(lambda s: tag_with_patterns(s, tarefa_patterns))
out['Metodo']   = txt.map(lambda s: tag_with_patterns(s, metodo_patterns))
out['Tecnica']  = txt.map(lambda s: tag_with_patterns(s, tecnica_patterns))
out['Dados']    = txt.map(lambda s: tag_with_patterns(s, dados_patterns))

out_path = (Path(RUN_DIR)/'artifacts'/'labels_multi_axis.csv')
out.to_csv(out_path, index=False)
cov_f = (out['Fenomeno'].fillna('')!='').mean()
cov_t = (out['Tecnica'].fillna('')!='').mean()
print('✅ labels_multi_axis.csv salvo em:', out_path)
print(f'Cobertura Fenômeno: {cov_f:.3f}')
print(f'Cobertura Técnica : {cov_t:.3f}')


In [ ]:
#@title 🔎 Coverage + top n-grams (missings)
import re, pandas as pd
from collections import Counter
from pathlib import Path

ART_DIR = Path(RUN_DIR) / "artifacts"
labs = pd.read_csv(ART_DIR/'labels_multi_axis.csv')

def tokenize(s):
    s = str(s or '').lower()
    s = re.sub(r'https?://\S+|doi:\S+',' ', s)
    s = re.sub(r'[^a-z0-9\-_/ ]+',' ', s)
    toks = [t for t in s.split() if len(t) > 2]
    return toks

def top_ngrams(series, n=30):
    unis, bis = Counter(), Counter()
    for s in series.dropna().tolist():
        toks = tokenize(s)
        for i,t in enumerate(toks):
            unis[t]+=1
            if i+1 < len(toks):
                bis[(t, toks[i+1])] += 1
    return unis.most_common(n), [(' '.join(k),v) for k,v in bis.most_common(n)]

# Precisamos dos textos fonte
try:
    dom = pd.concat([dfs['nucleo'], dfs['piml'], dfs['combf']], ignore_index=True)
except NameError:
    dom = pd.concat([
        pd.read_csv(ART_DIR/'concat_nucleo.csv'),
        pd.read_csv(ART_DIR/'concat_piml.csv'),
        pd.read_csv(ART_DIR/'concat_combf.csv'),
    ], ignore_index=True)

for c in ['TI','DE','ID','AB']:
    if c not in dom.columns: dom[c] = ''
dom['text_full'] = (dom['TI'].astype(str)+' '+dom['DE'].astype(str)+' '+dom['ID'].astype(str)+' '+dom['AB'].astype(str))

no_fen = labs[labs['Fenomeno'].fillna('')=='']
no_tec = labs[labs['Tecnica'].fillna('')=='']

print(f"Cobertura Fenômeno: {(labs['Fenomeno'].fillna('')!='').mean():.3f}")
print(f"Cobertura Técnica : {(labs['Tecnica'].fillna('')!='').mean():.3f}")

u1,b1 = top_ngrams(dom.loc[no_fen.index,'text_full'], 40)
u2,b2 = top_ngrams(dom.loc[no_tec.index,'text_full'], 40)

print("\nTOP unigrams (missing Fenomeno):")
for w,c in u1[:30]: print(f"{w:<24} {c}")
print("\nTOP bigrams (missing Fenomeno):")
for w,c in b1[:30]: print(f"{w:<30} {c}")

print("\nTOP unigrams (missing Tecnica):")
for w,c in u2[:30]: print(f"{w:<24} {c}")
print("\nTOP bigrams (missing Tecnica):")
for w,c in b2[:30]: print(f"{w:<30} {c}")


In [ ]:
  #@title Projeção no ML-108k + FAISS + consultas seed
from sentence_transformers import SentenceTransformer
import numpy as np, faiss, math
st = SentenceTransformer(str(CKPT_DIR / 'SciBERT-SolarPhysics-Search'))
ml_texts = dfs['ml']['text_full'].tolist()

B=256; vecs=[]
for i in range(0, len(ml_texts), B):
    vecs.append(st.encode(ml_texts[i:i+B], normalize_embeddings=True))
vecs = np.vstack(vecs).astype('float32')
index = faiss.IndexFlatIP(vecs.shape[1])
faiss.normalize_L2(vecs)
index.add(vecs)
faiss.write_index(index, str(ART_DIR/'faiss_index.bin'))
log_write('05_projection_ml.log', f'embed_ml | docs={len(ml_texts)} | dim={vecs.shape[1]}')

seed_queries = [
  'physics-informed transformer for PDEs in magnetohydrodynamics',
  'multimodal transformer with magnetograms and GOES time series',
  'PINN for plasma turbulence and solar flare forecasting'
]
out_rows = []
for q in seed_queries:
    qv = st.encode([q], normalize_embeddings=True).astype('float32')
    D, I = index.search(qv, 100)
    for rank, (score, idx) in enumerate(zip(D[0], I[0]), start=1):
        out_rows.append({'query':q, 'rank':rank, 'score':float(score), 'idx':int(idx)})
pd.DataFrame(out_rows).to_csv(REP_DIR/'seed_search_hits.csv', index=False)
print('✅ Índice FAISS gravado e consultas seed salvas')

In [ ]:
#@title Fenômeno × Técnica (co-ocorrência + PMI robusto)
import math
import numpy as np
import pandas as pd
from collections import Counter

labs = pd.read_csv(ART_DIR/'labels_multi_axis.csv')

# --- Normalização defensiva das colunas-alvo ---
for col in ['Fenomeno', 'Tecnica']:
    if col not in labs.columns:
        labs[col] = ''
    labs[col] = labs[col].fillna('').astype(str)

# tokens que devem ser ignorados (aparecem como texto após astype(str))
BAD_TOKENS = {'', 'nan', 'none', 'null'}

def explode_col(s: str):
    # s já é string; divide e limpa
    parts = [x.strip() for x in s.split(';')]
    return [x for x in parts if x and x.lower() not in BAD_TOKENS]

fen = labs['Fenomeno'].apply(explode_col)
tec = labs['Tecnica'].apply(explode_col)

# Se alguma coluna vier toda vazia, evita divisão por zero e salva CSV vazio informativo
N = len(labs)
if N == 0 or fen.map(len).sum() == 0 or tec.map(len).sum() == 0:
    out = ART_DIR/'fenomeno_x_tecnica.csv'
    pd.DataFrame(columns=['Fenomeno','Tecnica','freq','PMI']).to_csv(out, index=False)
    print('⚠️ Colunas vazias — arquivo salvo sem linhas em', out)
else:
    cf = Counter([x for lst in fen for x in lst])
    ct = Counter([x for lst in tec for x in lst])

    # co-ocorrência (todas as combinações f×t por linha)
    co = Counter()
    for f_list, t_list in zip(fen, tec):
        for a in f_list:
            for b in t_list:
                co[(a, b)] += 1

    rows = []
    EPS = 1e-9
    for (a, b), c in co.items():
        p_ab = c / N
        p_a = cf[a] / N
        p_b = ct[b] / N
        # PMI com suavização para evitar log(0)
        pmi = math.log((p_ab + EPS) / ((p_a * p_b) + EPS))
        rows.append({'Fenomeno': a, 'Tecnica': b, 'freq': c, 'PMI': pmi})

    df_out = (
        pd.DataFrame(rows)
        .sort_values(['PMI','freq'], ascending=[False, False])
        .reset_index(drop=True)
    )
    out = ART_DIR/'fenomeno_x_tecnica.csv'
    df_out.to_csv(out, index=False)
    print('✅ Tabela fenomeno_x_tecnica.csv salva em', out,
          f'| pares={len(df_out)} | N={N}')


In [ ]:
#@title Sumário final e próximos passos
print('Artefatos principais:')
for p in [
    ART_DIR/'concat_nucleo.csv', ART_DIR/'concat_piml.csv', ART_DIR/'concat_combf.csv', ART_DIR/'concat_ml.csv',
    CKPT_DIR/'scibert_dapt_best', CKPT_DIR/'SciBERT-SolarPhysics-Search',
    ART_DIR/'faiss_index.bin', ART_DIR/'fenomeno_x_tecnica.csv', REP_DIR/'clusters_summary.csv', REP_DIR/'seed_search_hits.csv'
]:
    print(' -', p)
print('\nLogs em:', LOG_DIR, '\nRelatórios em:', REP_DIR)

In [ ]:
#@title Compactar e baixar todos os resultados do pipeline
import shutil, zipfile, os
from google.colab import files

RUN_PATH = str(RUN_DIR)  # caminho do seu run atual
ZIP_PATH = '/content/SciBERT_SolarPhysics_Search_results.zip'

# compacta logs, ckpts, artifacts e reports
shutil.make_archive(ZIP_PATH.replace('.zip',''), 'zip', RUN_PATH)
print(f'✅ Arquivo gerado: {ZIP_PATH}')

# baixa para o seu computador
files.download(ZIP_PATH)


In [ ]:
#@title Carregar modelo final e índice FAISS
from sentence_transformers import SentenceTransformer, util
import faiss, numpy as np, pandas as pd
from pathlib import Path

ART_DIR = Path(RUN_DIR) / 'artifacts'
REP_DIR = Path(RUN_DIR) / 'reports'
CKPT_DIR = Path(RUN_DIR) / 'ckpts'

# modelo final
model_path = CKPT_DIR / 'SciBERT-SolarPhysics-Search'
model = SentenceTransformer(str(model_path))
print("✅ Modelo carregado de:", model_path)

# índice FAISS
index = faiss.read_index(str(ART_DIR/'faiss_index.bin'))
print("✅ Índice FAISS carregado:", index.ntotal, "documentos")


In [ ]:
#@title 🔎 Busca semântica manual (sem meta.parquet)
from sentence_transformers import SentenceTransformer
import faiss, numpy as np, pandas as pd
from pathlib import Path

ART_DIR = Path(RUN_DIR) / 'artifacts'
CKPT_DIR = Path(RUN_DIR) / 'ckpts'

# carrega modelo e índice
model = SentenceTransformer(str(CKPT_DIR / 'SciBERT-SolarPhysics-Search'))
index = faiss.read_index(str(ART_DIR / 'faiss_index.bin'))
print("✅ Modelo e índice carregados")

# tenta carregar metadados de fallback
try:
    meta = pd.concat([
        pd.read_csv(ART_DIR/'concat_nucleo.csv'),
        pd.read_csv(ART_DIR/'concat_piml.csv'),
        pd.read_csv(ART_DIR/'concat_combf.csv'),
        pd.read_csv(ART_DIR/'concat_ml.csv'),
    ], ignore_index=True)
except Exception as e:
    print("⚠️ Metadados originais não encontrados, usando labels_multi_axis.csv")
    meta = pd.read_csv(ART_DIR/'labels_multi_axis.csv')

# escolha seu campo de texto preferido
if 'AB' in meta.columns:
    texts = meta['AB'].fillna('').astype(str).tolist()
elif 'text_full' in meta.columns:
    texts = meta['text_full'].fillna('').astype(str).tolist()
else:
    texts = meta.astype(str).agg(' '.join, axis=1).tolist()

# consulta manual
query = "physics-informed transformer for solar flare prediction"  # altere
k = 10

vec = model.encode([query], normalize_embeddings=True).astype('float32')
scores, idxs = index.search(vec, k)

print(f"\n🔎 Top {k} resultados para:")
print(">", query, "\n")

for rank, (score, idx) in enumerate(zip(scores[0], idxs[0]), start=1):
    title = meta.iloc[idx]['TI'] if 'TI' in meta.columns else ''
    abstract = meta.iloc[idx]['AB'] if 'AB' in meta.columns else meta.iloc[idx]
    print(f"{rank}. [{score:.3f}] {title[:120]}")
    print(f"   {str(abstract)[:300]}...\n")


In [ ]:
#@title Relações Fenômeno × Técnica
df_pmi = pd.read_csv(ART_DIR/'fenomeno_x_tecnica.csv')
top = df_pmi.sort_values('PMI', ascending=False).head(25)
display(top)

# gráfico simples
import matplotlib.pyplot as plt
plt.figure(figsize=(10,6))
plt.barh(top['Fenomeno'] + " ↔ " + top['Tecnica'], top['PMI'])
plt.xlabel('PMI (força de associação)')
plt.title('Top 25 relações Fenômeno–Técnica')
plt.gca().invert_yaxis()
plt.show()


In [ ]:
#@title Redução UMAP + visualização de clusters (sem meta.parquet)
import umap, matplotlib.pyplot as plt
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sentence_transformers import SentenceTransformer
from pathlib import Path

ART_DIR = Path(RUN_DIR) / 'artifacts'
CKPT_DIR = Path(RUN_DIR) / 'ckpts'

# ✅ Carrega modelo
model = SentenceTransformer(str(CKPT_DIR / 'SciBERT-SolarPhysics-Search'))
print("Modelo carregado.")

# ✅ Carrega textos de fallback (concat CSVs)
parts = []
for name in ['concat_nucleo.csv','concat_piml.csv','concat_combf.csv','concat_ml.csv']:
    p = ART_DIR/name
    if p.exists():
        parts.append(pd.read_csv(p))
if not parts:
    raise FileNotFoundError("Nenhum concat_*.csv encontrado em artifacts/")
meta = pd.concat(parts, ignore_index=True)
print(f"Carregados {len(meta):,} registros")

# ✅ Seleciona campo de texto (abstract ou text_full)
if 'AB' in meta.columns:
    texts = meta['AB'].fillna('').astype(str)
elif 'text_full' in meta.columns:
    texts = meta['text_full'].fillna('').astype(str)
else:
    texts = meta.astype(str).agg(' '.join, axis=1)

# ✅ Amostragem (para performance)
N = min(5000, len(texts))
sample_idx = np.random.choice(len(texts), N, replace=False)
texts_sample = texts.iloc[sample_idx].tolist()

# ✅ Gera embeddings
print(f"Gerando embeddings para {N} textos…")
embs = model.encode(texts_sample, normalize_embeddings=True, show_progress_bar=True)

# ✅ Reduz dimensão e plota
umap_2d = umap.UMAP(n_neighbors=30, min_dist=0.1, metric='cosine').fit_transform(embs)
plt.figure(figsize=(9,7))
plt.scatter(umap_2d[:,0], umap_2d[:,1], s=4, alpha=0.6)
plt.title(f'Projeção 2D de {N} embeddings (amostra aleatória)')
plt.xlabel('UMAP-1'); plt.ylabel('UMAP-2')
plt.show()


In [ ]:
#@title Explorar artigos do cluster isolado (esquerda)
import numpy as np, pandas as pd

# pega as coordenadas UMAP calculadas antes
umap_df = pd.DataFrame(umap_2d, columns=['x','y'])
threshold = -10  # ajustável conforme seu gráfico

isolated_idx = umap_df[umap_df['x'] < threshold].index
print(f"{len(isolated_idx)} artigos no cluster isolado (x < {threshold})")

meta.iloc[sample_idx[isolated_idx]][['TI','AB']].head(10)


In [ ]:
#@title 🎨 UMAP colorido por Técnica (alinhando por DOI; fallback com regex)
import re, numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

ART_DIR = Path(RUN_DIR) / 'artifacts'

# 1) Garantir que 'meta' e 'sample_idx' existem (da célula UMAP anterior)
assert 'meta' in globals() and 'sample_idx' in globals(), "Execute antes a célula UMAP que cria meta, sample_idx e umap_2d."

# 2) Tentar alinhar labels -> meta por DOI (DI)
labels = pd.read_csv(ART_DIR/'labels_multi_axis.csv')
labs = labels.rename(columns={'DOI':'DI'})  # alinhar nome

# pode haver múltiplas linhas com mesmo DOI; manter a mais completa
labs = labs.drop_duplicates(subset=['DI'], keep='first')

# join por DI mantendo ordem do meta
if 'DI' in meta.columns:
    meta_labs = meta[['DI']].merge(labs[['DI','Tecnica']], on='DI', how='left')
else:
    # se não há DI, criamos coluna vazia e iremos ao fallback
    meta_labs = pd.DataFrame({'Tecnica': [np.nan]*len(meta)})

tecnica_series = meta_labs['Tecnica']

# 3) Fallback opcional: preencher NaN com regra leve de regex (quando labels faltam)
def soft_tag_tecnica(text):
    s = str(text).lower()
    if re.search(r'\bvision\s+transformer\b|\bvit\b|\bswin[-\s]?transformer\b', s): return 'vit_vision_transformer'
    if re.search(r'\btransformer(s)?\b|\bself[-\s]?attention\b|\bcross[-\s]?attention\b', s): return 'transformer'
    if re.search(r'\bmultimodal\b|multi[-\s]?modal|data\s+fusion|feature\s+fusion', s): return 'multimodal_fusion'
    if re.search(r'\bcnn(s)?\b|convolutional\s+neural', s): return 'cnn'
    if re.search(r'\blstm\b|\bgru\b|\brnn\b|recurrent\s+neural', s): return 'lstm_gru_rnn'
    if re.search(r'\bgnn\b|graph\s+neural|graph\s+convolution', s): return 'gnn_graph'
    if re.search(r'\bsvm\b|support\s+vector\s+machine|random\s+forest|xgboost|\bxgb\b', s): return 'classic_ml'
    if re.search(r'\bautoencoder\b|variational\s+autoencoder|\bvae\b', s): return 'autoencoder_vae'
    if re.search(r'\bdeep\s+learning\b', s): return 'deep_learning'
    if re.search(r'\bmachine\s+learning\b', s): return 'machine_learning'
    return 'none'

if tecnica_series.isna().any():
    # tentar usar Abstract se existir; senão, text_full; senão concatena tudo
    if 'AB' in meta.columns:
        base_text = meta['AB'].fillna('')
    elif 'text_full' in meta.columns:
        base_text = meta['text_full'].fillna('')
    else:
        base_text = meta.astype(str).agg(' '.join, axis=1)
    tecnica_fallback = base_text.apply(soft_tag_tecnica)
    tecnica_series = tecnica_series.fillna(tecnica_fallback)

# 4) Gerar vetor de cores somente para os pontos amostrados
tecnica_codes = tecnica_series.astype('category').cat.codes.values
colors = tecnica_codes[sample_idx]

# 5) Plot
plt.figure(figsize=(9,7))
sc = plt.scatter(umap_2d[:,0], umap_2d[:,1], c=colors, cmap='tab20', s=6, alpha=0.75)
plt.title('UMAP colorido por Técnica (alinhado por DOI; fallback regex)')
plt.xlabel('UMAP-1'); plt.ylabel('UMAP-2')

# legenda: mapear índice de categoria -> rótulo
cats = tecnica_series.astype('category').cat.categories
# mostrar só as 12 mais frequentes para a legenda não poluir
top_counts = pd.Series(tecnica_series).value_counts().head(12).index.tolist()
legend_items = [c for c in cats if c in top_counts]
from matplotlib.lines import Line2D
handles = []
for lab in legend_items:
    code = list(cats).index(lab)
    handles.append(Line2D([0],[0], marker='o', color='w',
                          label=lab, markerfacecolor=plt.cm.tab20(code%20), markersize=8))
plt.legend(handles=handles, title='Técnica (top)', bbox_to_anchor=(1.02,1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
#@title Evolução temporal de técnicas
labels = pd.read_csv(ART_DIR/'labels_multi_axis.csv')
labels['PY'] = pd.to_numeric(labels['PY'], errors='coerce')

trend = (labels.assign(Tecnica=labels['Tecnica'].str.split(';'))
         .explode('Tecnica')
         .groupby(['PY','Tecnica']).size().reset_index(name='count'))
top_tec = trend.groupby('Tecnica')['count'].sum().nlargest(8).index
subset = trend[trend['Tecnica'].isin(top_tec)]

plt.figure(figsize=(10,6))
for t in top_tec:
    s = subset[subset['Tecnica']==t]
    plt.plot(s['PY'], s['count'], marker='o', label=t)
plt.legend()
plt.title('Tendência temporal das principais técnicas (2020–2025)')
plt.xlabel('Ano'); plt.ylabel('Nº de artigos')
plt.show()


In [ ]:
#@title Comparar embeddings de conceitos
queries = [
    "solar flare prediction with physics-informed neural networks",
    "magnetic reconnection using transformers",
    "plasma turbulence analysis with graph neural networks",
]
vecs = model.encode(queries, normalize_embeddings=True)
sim = util.cos_sim(vecs, vecs)
pd.DataFrame(sim.numpy(), index=queries, columns=queries).round(3)


### Comparação entre SciBERT X SciBERT-SolarPhysics-Search

In [ ]:
#@title Carregar textos (concat_*.csv) + rótulos fracos + preparar amostra
import pandas as pd, numpy as np
from pathlib import Path

ART_DIR = Path(RUN_DIR)/'artifacts'
REP_DIR = Path(RUN_DIR)/'reports'
REP_DIR.mkdir(parents=True, exist_ok=True)

# 1) Carrega metadados de todos os concat_*.csv (Núcleo/PIML/CombFinal/ML)
parts = []
for name in ['concat_nucleo.csv','concat_piml.csv','concat_combf.csv','concat_ml.csv']:
    p = ART_DIR/name
    if p.exists():
        parts.append(pd.read_csv(p))
if not parts:
    raise FileNotFoundError("Nenhum concat_*.csv encontrado em artifacts/")
meta = pd.concat(parts, ignore_index=True)

# 2) Texto preferido
txt_col = 'AB' if 'AB' in meta.columns else ('text_full' if 'text_full' in meta.columns else None)
if txt_col is None:
    meta['text_full'] = meta.astype(str).agg(' '.join, axis=1)
    txt_col = 'text_full'
meta[txt_col] = meta[txt_col].fillna('').astype(str)

# 3) Rótulos fracos (Fenômeno/Técnica)
labels = pd.read_csv(ART_DIR/'labels_multi_axis.csv')
labs = labels.rename(columns={'DOI':'DI'}).drop_duplicates(subset=['DI'], keep='first')
if 'DI' in meta.columns:
    meta = meta.merge(labs[['DI','Fenomeno','Tecnica']], on='DI', how='left')
else:
    meta[['Fenomeno','Tecnica']] = ''

# 4) Mantém apenas onde temos alguma classe de Técnica para avaliação
meta['Tecnica_clean'] = meta['Tecnica'].fillna('').str.split(';').str[0].fillna('')
keep = meta['Tecnica_clean']!=''
meta_eval = meta[keep].copy()

# 5) Amostra para acelerar (altere N se quiser)
N = min(8000, len(meta_eval))
rng = np.random.default_rng(42)
sample_idx = rng.choice(meta_eval.index, N, replace=False)
df_eval = meta_eval.loc[sample_idx].reset_index(drop=True)

print(f"Total para avaliação: {len(df_eval):,} docs com Técnica anotada")
print("Top 10 classes de Técnica:", df_eval['Tecnica_clean'].value_counts().head(10).to_dict())

# 6) Texto final da amostra
texts = df_eval[txt_col].tolist()
y_tecnica = df_eval['Tecnica_clean'].tolist()


In [ ]:
#@title Carregar modelos: Baseline SciBERT vs SciBERT-SolarPhysics-Search
from sentence_transformers import SentenceTransformer, models

# (A) Baseline: SciBERT "puro" como sentence-encoder (Transformer + Pooling + Normalize)
base_word = models.Transformer('allenai/scibert_scivocab_uncased')
base_pool = models.Pooling(base_word.get_word_embedding_dimension(), pooling_mode_mean_tokens=True)
base_norm = models.Normalize()
model_baseline = SentenceTransformer(modules=[base_word, base_pool, base_norm])
print("✅ Baseline pronto.")

# (B) Seu modelo fine-tunado
from pathlib import Path
CKPT_DIR = Path(RUN_DIR)/'ckpts'
model_ft = SentenceTransformer(str(CKPT_DIR/'SciBERT-SolarPhysics-Search'))
print("✅ Seu modelo carregado:", CKPT_DIR/'SciBERT-SolarPhysics-Search')


In [ ]:
#@title Clusterização: NMI / ARI / Silhouette (Técnica)
import numpy as np
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score, silhouette_score
from sklearn.preprocessing import LabelEncoder
from sklearn.cluster import KMeans

le = LabelEncoder()
y_true = le.fit_transform(y_tecnica)
n_classes = len(le.classes_)

def eval_clustering(embs, y_true, name, k=None):
    # KMeans com K = nº de classes observadas (proxy)
    k = k or len(np.unique(y_true))
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    y_pred = km.fit_predict(embs)
    nmi = normalized_mutual_info_score(y_true, y_pred)
    ari = adjusted_rand_score(y_true, y_pred)
    # Silhouette em subamostra (para acelerar)
    sub = min(5000, len(embs))
    sil = silhouette_score(embs[:sub], y_pred[:sub], metric='cosine')
    return {'model': name, 'k': k, 'NMI': nmi, 'ARI': ari, 'Silhouette': sil}

# Embeddings (pode levar alguns minutos)
emb_base = model_baseline.encode(texts, normalize_embeddings=True, batch_size=128, show_progress_bar=True)
emb_ft   = model_ft.encode(texts, normalize_embeddings=True, batch_size=128, show_progress_bar=True)

res_base = eval_clustering(emb_base, y_true, 'SciBERT-baseline')
res_ft   = eval_clustering(emb_ft,   y_true, 'SciBERT-SolarPhysics-Search')

import pandas as pd
df_clu = pd.DataFrame([res_base, res_ft])
df_clu.to_csv(REP_DIR/'eval_clustering_tecnica.csv', index=False)
df_clu


In [ ]:
#@title  Retrieval por classe de Técnica (Recall@K / MRR)
import faiss, numpy as np, pandas as pd
from collections import defaultdict

def build_index(x):
    x = x.astype('float32').copy()
    faiss.normalize_L2(x)
    index = faiss.IndexFlatIP(x.shape[1])
    index.add(x)
    return index

def eval_retrieval(embs, labels, Ks=(10, 50, 100)):
    x = np.asarray(embs, dtype='float32')
    index = build_index(x)
    D, I = index.search(x, max(Ks)+1)  # +1 por causa do próprio item
    # remove o próprio item da busca (rank 0)
    D, I = D[:,1:], I[:,1:]
    label_arr = np.array(labels)
    rr_list = []
    recalls = defaultdict(list)
    for i in range(len(labels)):
        target = label_arr[i]
        hits = (label_arr[I[i]] == target)
        # RR
        hit_positions = np.where(hits)[0]
        rr_list.append(1.0/(hit_positions[0]+1) if len(hit_positions)>0 else 0.0)
        # Recall@K
        for K in Ks:
            recalls[K].append(hits[:K].any())
    out = {'MRR': float(np.mean(rr_list))}
    for K in Ks:
        out[f'Recall@{K}'] = float(np.mean(recalls[K]))
    return out

ret_base = eval_retrieval(emb_base, y_tecnica, Ks=(10,50,100))
ret_ft   = eval_retrieval(emb_ft,   y_tecnica, Ks=(10,50,100))

df_ret = pd.DataFrame([
    {'model':'SciBERT-baseline', **ret_base},
    {'model':'SciBERT-SolarPhysics-Search', **ret_ft}
])
df_ret.to_csv(REP_DIR/'eval_retrieval_tecnica.csv', index=False)
df_ret


In [ ]:
#@title UMAP lado a lado (baseline vs fine-tuned)
import umap, matplotlib.pyplot as plt

# Redução para 2D
um_base = umap.UMAP(n_neighbors=30, min_dist=0.1, metric='cosine', random_state=42).fit_transform(emb_base)
um_ft   = umap.UMAP(n_neighbors=30, min_dist=0.1, metric='cosine', random_state=42).fit_transform(emb_ft)

fig, ax = plt.subplots(1,2, figsize=(13,6))
ax[0].scatter(um_base[:,0], um_base[:,1], s=4, alpha=0.6)
ax[0].set_title('SciBERT baseline — UMAP')
ax[0].set_xlabel('UMAP-1'); ax[0].set_ylabel('UMAP-2')

ax[1].scatter(um_ft[:,0], um_ft[:,1], s=4, alpha=0.6, color='tab:blue')
ax[1].set_title('SciBERT-SolarPhysics-Search — UMAP')
ax[1].set_xlabel('UMAP-1'); ax[1].set_ylabel('UMAP-2')

plt.tight_layout()
fig_path = REP_DIR/'umap_baseline_vs_ft.png'
plt.savefig(fig_path, dpi=160)
print("✅ Figura salva em:", fig_path)
plt.show()


In [ ]:
#@title Separabilidade por classe (nearest-centroid)
import numpy as np, pandas as pd
from sklearn.metrics import accuracy_score

def nearest_centroid_accuracy(embs, labels):
    labels = np.array(labels)
    classes = np.unique(labels)
    cent = {}
    for c in classes:
        cent[c] = np.mean(embs[labels==c], axis=0)
    cent_mat = np.vstack([cent[c] for c in classes]).astype('float32')
    # cosine sim
    em = np.asarray(embs, dtype='float32')
    em_norm = em/np.linalg.norm(em, axis=1, keepdims=True)
    cent_norm = cent_mat/np.linalg.norm(cent_mat, axis=1, keepdims=True)
    sim = em_norm @ cent_norm.T
    pred = classes[np.argmax(sim, axis=1)]
    acc = accuracy_score(labels, pred)
    return acc

acc_base = nearest_centroid_accuracy(emb_base, np.array(y_tecnica))
acc_ft   = nearest_centroid_accuracy(emb_ft,   np.array(y_tecnica))

pd.DataFrame([
    {'model':'SciBERT-baseline', 'NearestCentroidAcc': acc_base},
    {'model':'SciBERT-SolarPhysics-Search', 'NearestCentroidAcc': acc_ft}
]).to_csv(REP_DIR/'eval_nearest_centroid.csv', index=False)
print(f"Baseline Nearest-Centroid Acc: {acc_base:.3f}")
print(f"Fine-tuned Nearest-Centroid Acc: {acc_ft:.3f}")


In [ ]:
#@title Relatório rápido (Markdown) com os resultados
from pathlib import Path
rep_md = Path(REP_DIR)/'AB_experimento_baseline_vs_finetuned.md'

df_clu = pd.read_csv(REP_DIR/'eval_clustering_tecnica.csv')
df_ret = pd.read_csv(REP_DIR/'eval_retrieval_tecnica.csv')
df_nc  = pd.read_csv(REP_DIR/'eval_nearest_centroid.csv')

with open(rep_md, 'w') as f:
    f.write("# Experimento A/B — SciBERT (baseline) vs SciBERT-SolarPhysics-Search\n\n")
    f.write("## Clusterização (Técnica)\n")
    f.write(df_clu.to_markdown(index=False)); f.write("\n\n")
    f.write("## Retrieval por classe (Técnica)\n")
    f.write(df_ret.to_markdown(index=False)); f.write("\n\n")
    f.write("## Acurácia por centróide (Técnica)\n")
    f.write(df_nc.to_markdown(index=False)); f.write("\n\n")
    f.write("![UMAP](umap_baseline_vs_ft.png)\n")
print("✅ Relatório salvo em:", rep_md)


In [ ]:
#@title 📝 Relatório rápido (exibir no notebook + salvar Markdown)
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

rep_md = Path(REP_DIR)/'AB_experimento_baseline_vs_finetuned.md'

# lê resultados
df_clu = pd.read_csv(REP_DIR/'eval_clustering_tecnica.csv')
df_ret = pd.read_csv(REP_DIR/'eval_retrieval_tecnica.csv')
df_nc  = pd.read_csv(REP_DIR/'eval_nearest_centroid.csv')

# grava em markdown
with open(rep_md, 'w') as f:
    f.write("# Experimento A/B — SciBERT (baseline) vs SciBERT-SolarPhysics-Search\n\n")
    f.write("## Clusterização (Técnica)\n")
    f.write(df_clu.to_markdown(index=False)); f.write("\n\n")
    f.write("## Retrieval por classe (Técnica)\n")
    f.write(df_ret.to_markdown(index=False)); f.write("\n\n")
    f.write("## Acurácia por centróide (Técnica)\n")
    f.write(df_nc.to_markdown(index=False)); f.write("\n\n")
    f.write("![UMAP](umap_baseline_vs_ft.png)\n")

print("✅ Relatório salvo em:", rep_md)

# -------- Mostra diretamente no Colab --------
display(Markdown("### 🧭 *Experimento A/B — SciBERT (baseline) vs Fine-tuned*\n"))

display(Markdown("#### **Clusterização (Técnica)**"))
display(df_clu.style.background_gradient(cmap='Blues'))

display(Markdown("#### **Retrieval por classe (Técnica)**"))
display(df_ret.style.background_gradient(cmap='Greens'))

display(Markdown("#### **Acurácia por centróide (Técnica)**"))
display(df_nc.style.background_gradient(cmap='Oranges'))

from IPython.display import Image
img_path = REP_DIR/'umap_baseline_vs_ft.png'
if img_path.exists():
    display(Markdown("#### **Comparativo UMAP (Baseline vs Fine-tuned)**"))
    display(Image(filename=str(img_path)))
else:
    print("⚠️ Figura UMAP não encontrada em:", img_path)


In [ ]:
# Comparativo de ganhos (delta %)
a = df_clu.set_index('model')
def rel(a,b): return (b-a)/max(a,1e-9)
display(Markdown(
    f"**ΔNMI:** {100*rel(a.loc['SciBERT-baseline','NMI'],a.loc['SciBERT-SolarPhysics-Search','NMI']):.1f}% &nbsp;&nbsp;"
    f"ΔARI: {100*rel(a.loc['SciBERT-baseline','ARI'],a.loc['SciBERT-SolarPhysics-Search','ARI']):.1f}% &nbsp;&nbsp;"
    f"ΔSilhouette: {100*rel(a.loc['SciBERT-baseline','Silhouette'],a.loc['SciBERT-SolarPhysics-Search','Silhouette']):.1f}%"
))


### Publica Huggingface

In [ ]:
# NOTE: This cell is part of the end-to-end pipeline. Read the markdown cells for context and required inputs.
from huggingface_hub import login
login()


In [ ]:
# NOTE: This cell is part of the end-to-end pipeline. Read the markdown cells for context and required inputs.
import os
for root, dirs, files in os.walk("/content"):
    if "model.safetensors" in files:
        print("✅ Modelo encontrado em:", root)
        break


In [ ]:
# 2) Caminho CORRETO do seu modelo
import os, glob
model_path = "/content/runs/20251105_203842_SciBERT_SolarPhysics/ckpts/SciBERT-SolarPhysics-Search"

# sanity check
print("Existe?", os.path.isdir(model_path))
print("Arquivos:", sorted(os.listdir(model_path)))
assert os.path.isfile(os.path.join(model_path, "model.safetensors")), "model.safetensors não encontrado"
assert os.path.isfile(os.path.join(model_path, "config.json")), "config.json não encontrado"
assert os.path.isfile(os.path.join(model_path, "modules.json")), "modules.json não encontrado"

# 3) Carregar e publicar
from sentence_transformers import SentenceTransformer
st_model = SentenceTransformer(model_path)  # entende .safetensors
st_model.push_to_hub(
    "andreinsardi/SciBERT-SolarPhysics-Search",  # troque pelo seu user se for outro
    commit_message="Initial release of SciBERT-SolarPhysics-Search (fine-tuned for solar physics)"
)


In [ ]:
# PIPE 1 — ZIP + DOWNLOAD do corpus base (4x RData)

from pathlib import Path
from datetime import datetime, timezone
import zipfile
from google.colab import files

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

CORPUS_DIR = Path("/content/data/corpus_rdata")
ZIP_PATH = Path(f"/content/PIPE1_corpus_rdata_{RUN_ID}.zip")

required = [
    "CombFinal_bibliometrix.RData",
    "ML_Scopus_bibliometrix.RData",
    "Nucleo_bibliometrix.RData",
    "PIML_bibliometrix.RData",
]

# valida
missing = [f for f in required if not (CORPUS_DIR / f).exists()]
if missing:
    raise FileNotFoundError(
        "❌ Arquivos ausentes no PIPE 1:\n" +
        "\n".join([str(CORPUS_DIR / m) for m in missing])
    )

# cria zip
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for fname in required:
        z.write(CORPUS_DIR / fname, arcname=fname)

print(f"[PIPE1] ZIP criado em: {ZIP_PATH}")

# download
files.download(str(ZIP_PATH))


In [ ]:
# ✅ ÚNICA CÉLULA — Completar o PIPE 1 a partir da sessão (runs/artefacts), salvar em pasta separada e FAZER DOWNLOAD
# Gera/Exporta (se faltar):
#  1) faiss_index.bin        (copia se existir)
#  2) docs_master.csv        (copia ou reconstrói)
#  3) index_map.csv          (gera: faiss_pos ↔ doc_id)
#  4) ckpts/                 (copia se existir)
#
# 👉 Basta rodar esta célula no Colab na mesma sessão onde existe /content/runs/...

import os, re, json, shutil, zipfile, hashlib
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

# download no Colab
try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
BASE = Path("/content")
RUNS_DIR = BASE / "runs"

OUT_DIR = BASE / f"PIPE1_missing_export_{RUN_ID}"
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "ckpts").mkdir(exist_ok=True)

def log(msg):
    print(f"[PIPE1-FIX][{RUN_ID}] {msg}")

def find_latest_session_folder():
    if not RUNS_DIR.exists():
        return None
    # prioriza pastas com "SciBERT_SolarPhysics" no nome
    candidates = [p for p in RUNS_DIR.iterdir() if p.is_dir() and "SciBERT" in p.name]
    if not candidates:
        candidates = [p for p in RUNS_DIR.iterdir() if p.is_dir()]
    if not candidates:
        return None
    # escolhe a mais recente por mtime
    candidates.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return candidates[0]

def deep_find(root: Path, patterns):
    """Procura arquivos por padrões (regex ou endswith) dentro de root."""
    hits = []
    if not root or not root.exists():
        return hits
    for p in root.rglob("*"):
        if not p.is_file():
            continue
        name = p.name
        for pat in patterns:
            if isinstance(pat, str) and name.lower() == pat.lower():
                hits.append(p); break
            if hasattr(pat, "search") and pat.search(name):
                hits.append(p); break
    return hits

def pick_best(hits):
    if not hits:
        return None
    # escolhe o maior (geralmente o índice FAISS/CSV principal)
    hits.sort(key=lambda p: p.stat().st_size, reverse=True)
    return hits[0]

# 1) localizar sessão
session = find_latest_session_folder()
if session is None:
    raise FileNotFoundError("❌ Não encontrei /content/runs/<sessão>. Rode o Pipe 1 primeiro ou extraia o ZIP de resultados antes.")

log(f"Encontrada sessão candidata: {session}")

# 2) localizar arquivos essenciais dentro da sessão
# Possíveis nomes para o índice
faiss_hits = deep_find(session, [
    re.compile(r"faiss.*\.(bin|index)$", re.I),
    re.compile(r".*\.faiss$", re.I),
])
faiss_path = pick_best(faiss_hits)

# Possíveis nomes para docs/corpus
docs_hits = deep_find(session, [
    re.compile(r"docs.*\.(csv|parquet)$", re.I),
    re.compile(r".*corpus.*\.(csv|parquet)$", re.I),
    re.compile(r".*concat.*\.(csv|parquet)$", re.I),
    re.compile(r".*records.*\.(csv|parquet)$", re.I),
])
docs_path = pick_best(docs_hits)

# ckpts
ckpt_dirs = [p for p in session.rglob("*") if p.is_dir() and p.name.lower() in ["ckpt","ckpts","checkpoints","checkpoint","model","models"]]
ckpt_dir = ckpt_dirs[0] if ckpt_dirs else None

# 3) copiar / reconstruir faiss_index.bin
if faiss_path:
    dst = OUT_DIR / "faiss_index.bin"
    shutil.copy2(faiss_path, dst)
    log(f"✅ Copiado FAISS index: {faiss_path} -> {dst}")
else:
    log("⚠️ Não achei FAISS index na sessão. (Você precisará exportar o índice no Pipe 1 original.)")

# 4) copiar / reconstruir docs_master.csv
docs_master = OUT_DIR / "docs_master.csv"

def load_docs_any(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path)

def normalize_docs(df: pd.DataFrame) -> pd.DataFrame:
    # tenta padronizar colunas mínimas
    cols = {c.lower(): c for c in df.columns}

    def colpick(cands):
        for c in cands:
            if c in cols: return cols[c]
        # contains fallback
        for c in df.columns:
            cl = c.lower()
            if any(x in cl for x in cands): return c
        return None

    doc_id_col = colpick(["doc_id","docid","paper_id","document_id","id","doi"])
    title_col  = colpick(["title","paper_title"])
    abs_col    = colpick(["abstract","summary","text"])
    year_col   = colpick(["year","pub_year","publication_year"])
    doi_col    = colpick(["doi"])
    src_col    = colpick(["source","database","db","origin"])

    # cria colunas padrão se existirem
    out = pd.DataFrame()
    if doc_id_col: out["doc_id"] = df[doc_id_col].astype(str)
    else: out["doc_id"] = [f"doc_{i}" for i in range(len(df))]

    out["title"] = df[title_col].astype(str) if title_col else ""
    out["abstract"] = df[abs_col].astype(str) if abs_col else ""
    out["year"] = df[year_col] if year_col else ""
    out["doi"] = df[doi_col].astype(str) if doi_col else ""
    out["source"] = df[src_col].astype(str) if src_col else ""

    # limpar "nan"
    for c in ["title","abstract","doi","source"]:
        out[c] = out[c].replace("nan","").fillna("")
    return out

if docs_path:
    try:
        df_raw = load_docs_any(docs_path)
        df_norm = normalize_docs(df_raw)
        df_norm.to_csv(docs_master, index=False)
        log(f"✅ docs_master.csv criado a partir de: {docs_path} ({len(df_norm)} docs)")
    except Exception as e:
        log(f"⚠️ Falha ao ler/reformatar {docs_path}: {e}")
else:
    log("⚠️ Não achei CSV/Parquet de docs na sessão. Vou tentar reconstruir a partir de arquivos concat_*.csv em /content...")
    # fallback: procurar concat_* diretamente no /content
    concat_hits = [p for p in BASE.glob("concat_*.csv")] + [p for p in BASE.glob("*concat*.csv")]
    if concat_hits:
        concat_hits.sort(key=lambda p: p.stat().st_size, reverse=True)
        df_raw = pd.read_csv(concat_hits[0])
        df_norm = normalize_docs(df_raw)
        df_norm.to_csv(docs_master, index=False)
        log(f"✅ docs_master.csv reconstruído de: {concat_hits[0]}")
    else:
        log("❌ Não consegui reconstruir docs_master.csv. Você precisa garantir que o Pipe 1 exporte um CSV com metadados+abstract.")
        # não aborta, para ainda baixar o que achou

# 5) gerar index_map.csv (faiss_pos ↔ doc_id) se houver docs_master
index_map = OUT_DIR / "index_map.csv"
if docs_master.exists():
    dfm = pd.read_csv(docs_master)
    dfm["faiss_pos"] = range(len(dfm))
    dfm[["faiss_pos","doc_id"]].to_csv(index_map, index=False)
    log(f"✅ index_map.csv gerado ({len(dfm)} linhas)")
else:
    log("⚠️ Não gerei index_map.csv porque docs_master.csv não existe.")

# 6) copiar ckpts (se houver)
if ckpt_dir:
    # copia tudo (mas limita tamanho em casos absurdos)
    dst_ck = OUT_DIR / "ckpts"
    # limpa dst_ck
    if dst_ck.exists():
        shutil.rmtree(dst_ck)
    shutil.copytree(ckpt_dir, dst_ck)
    log(f"✅ ckpts copiados: {ckpt_dir} -> {dst_ck}")
else:
    log("⚠️ Não encontrei pasta de checkpoints na sessão. (Se o Pipe 3 for usar seu encoder, você deve exportar ckpts.)")

# 7) criar manifest e zip
manifest = {
    "run_id": RUN_ID,
    "source_session": str(session),
    "export_dir": str(OUT_DIR),
    "files": {
        "faiss_index": "faiss_index.bin" if (OUT_DIR/"faiss_index.bin").exists() else None,
        "docs_master": "docs_master.csv" if docs_master.exists() else None,
        "index_map": "index_map.csv" if index_map.exists() else None,
        "ckpts_dir": "ckpts/" if (OUT_DIR/"ckpts").exists() else None,
    }
}
(Path(OUT_DIR) / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
log("✅ manifest.json criado")

ZIP_PATH = BASE / f"PIPE1_missing_export_{RUN_ID}.zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT_DIR.rglob("*"):
        if p.is_file():
            z.write(p, arcname=p.relative_to(OUT_DIR))
log(f"✅ ZIP criado: {ZIP_PATH}")

# 8) download
if IN_COLAB:
    files.download(str(ZIP_PATH))
else:
    log("ℹ️ Não detectei Colab; ZIP pronto em /content. Baixe manualmente.")


In [ ]:
# DOWNLOAD RECURSIVO — um arquivo por vez (Pipe 1 completo)

from pathlib import Path
from google.colab import files

# 👉 ajuste apenas este caminho se o nome do run mudar
ROOT_DIR = Path("/content/PIPE1_missing_export_20260114_225122")

if not ROOT_DIR.exists():
    raise FileNotFoundError(f"❌ Pasta não encontrada: {ROOT_DIR}")

print(f"📂 Iniciando download recursivo em: {ROOT_DIR}\n")

all_files = sorted([p for p in ROOT_DIR.rglob("*") if p.is_file()])

print(f"🔎 Total de arquivos encontrados: {len(all_files)}\n")

for p in all_files:
    print(f"⬇️  Baixando: {p.relative_to(ROOT_DIR)}")
    files.download(str(p))

print("\n✅ Download individual de todos os arquivos iniciado.")


In [ ]:
# NOTE: This cell is part of the end-to-end pipeline. Read the markdown cells for context and required inputs.
print("teste")